In [29]:
import pandas as pd
import yaml
import os
import pandas_ta as ta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import numpy as np

In [30]:
def load_config(config_path="../config/config.yaml"):
    config_path= os.path.abspath(config_path)
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config
config = load_config()
config

{'symbol': 'OGDC',
 'data': {'file_path': 'data/raw/',
  'columns': {'date': 'Date',
   'open': 'Open',
   'high': 'High',
   'low': 'Low',
   'close': 'Close',
   'volume': 'Volume'}},
 'model': {'target_column': 'Close', 'test_size': 0.2, 'shuffle': False},
 'features': {'sma': [9, 21, 50, 150, 200],
  'rsi': [14],
  'lag_days': [1, 2, 3]}}

In [31]:
def load_data():
    file_path = os.path.abspath(os.path.join("..", *config["data"]["file_path"].split("/"), config["symbol"] + ".csv"))
    df = pd.read_csv(file_path, index_col=False)
    return df
df = load_data()
df

,Date,Open,High,Low,Close,Volume
0,2004-01-19,52.60,53.40,52.00,52.05,31456000.0
1,2004-01-20,52.05,51.85,51.00,51.45,13285500.0
2,2004-01-21,51.45,53.40,51.55,53.10,46880000.0
3,2004-01-22,53.10,53.85,53.05,53.10,32411100.0
4,2004-01-23,53.10,53.40,52.50,52.65,19969400.0
...,...,...,...,...,...,...
5418,2026-01-29,335.01,336.00,320.16,321.81,10913390.0
5419,2026-01-30,325.00,329.50,318.60,323.93,6707829.0
5420,2026-02-02,323.79,330.98,320.26,328.97,5477804.0
5421,2026-02-03,328.97,332.50,327.00,329.43,3603595.0


# Data Exploration

### 1. Shape

In [32]:
df.shape

(5423, 6)

### 2. Missing Values


In [33]:
missings = df.isnull().sum(axis=0).to_frame(name="Missing values")
missings

,Missing values
Date,0
Open,0
High,0
Low,0
Close,0
Volume,0


### 3. Data types

In [34]:
dtyp = df.dtypes.to_frame(name="Dtype")
dtyp

,Dtype
Date,str
Open,float64
High,float64
Low,float64
Close,float64
Volume,float64


### 4. Duplicated rows

In [35]:
dup_col = int(df.T.duplicated().sum())

In [36]:
pd.concat((dtyp, missings), axis="columns")

,Dtype,Missing values
Date,str,0
Open,float64,0
High,float64,0
Low,float64,0
Close,float64,0
Volume,float64,0


### 5. Columns names

In [37]:
df.columns.to_list()

['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

# Data Preprocessing

### 1. Rename Columns

In [38]:
def col_rename(df):
    # Original Columns Names:
    col_date = config["data"]["columns"]["date"]
    col_open = config["data"]["columns"]["open"]
    col_high = config["data"]["columns"]["high"]
    col_low = config["data"]["columns"]["low"]
    col_close = config["data"]["columns"]["close"]
    col_volume = config["data"]["columns"]["volume"]
    
    # Renaming
    all_cols = {
        col_date: "Date",
        col_open: "Open",
        col_high: "High",
        col_low: "Low",
        col_close: "Close",
        col_volume: "Volume"
        }
    
    df = df[all_cols.keys()]
    df = df.rename(columns=all_cols)

    return df
col_rename(df)



,Date,Open,High,Low,Close,Volume
0,2004-01-19,52.60,53.40,52.00,52.05,31456000.0
1,2004-01-20,52.05,51.85,51.00,51.45,13285500.0
2,2004-01-21,51.45,53.40,51.55,53.10,46880000.0
3,2004-01-22,53.10,53.85,53.05,53.10,32411100.0
4,2004-01-23,53.10,53.40,52.50,52.65,19969400.0
...,...,...,...,...,...,...
5418,2026-01-29,335.01,336.00,320.16,321.81,10913390.0
5419,2026-01-30,325.00,329.50,318.60,323.93,6707829.0
5420,2026-02-02,323.79,330.98,320.26,328.97,5477804.0
5421,2026-02-03,328.97,332.50,327.00,329.43,3603595.0


### 2. Convert to datetime and sort

In [39]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values('Date')
df.dtypes


Date      datetime64[us]
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
dtype: object

### 3. Remove duplicate rows

In [40]:
df = df.drop_duplicates(keep='last')
print(df.duplicated().sum())

0


### 4. Drop missing values

In [41]:
df = df.dropna()
df.isna().sum()

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

# Feature Engineering

### 1. Percentage of price change

In [42]:
df["pct_change"] = df["Close"].pct_change().round(2)
df.head()

,Date,Open,High,Low,Close,Volume,pct_change
0,2004-01-19,52.60,53.40,52.00,52.05,31456000.0,NaN
1,2004-01-20,52.05,51.85,51.00,51.45,13285500.0,-0.01
2,2004-01-21,51.45,53.40,51.55,53.10,46880000.0,0.03
3,2004-01-22,53.10,53.85,53.05,53.10,32411100.0,0.00
4,2004-01-23,53.10,53.40,52.50,52.65,19969400.0,-0.01


### 2. Lag Features (previous day prices)

In [43]:
periods = config.get("features", {}).get("lag_days", [1, 2, 3])
for period in periods:
    df[f"Lag{period}"] = df["Close"].shift(periods=period)
df.head()

,Date,Open,High,Low,Close,Volume,pct_change,Lag1,Lag2,Lag3
0,2004-01-19,52.60,53.40,52.00,52.05,31456000.0,NaN,NaN,NaN,NaN
1,2004-01-20,52.05,51.85,51.00,51.45,13285500.0,-0.01,52.05,NaN,NaN
2,2004-01-21,51.45,53.40,51.55,53.10,46880000.0,0.03,51.45,52.05,NaN
3,2004-01-22,53.10,53.85,53.05,53.10,32411100.0,0.00,53.10,51.45,52.05
4,2004-01-23,53.10,53.40,52.50,52.65,19969400.0,-0.01,53.10,53.10,51.45


### 3. Moving Averages

In [44]:
periods = config.get("features", {}).get("sma", [20, 50, 100])
for period in periods:
    df[f'MA_{period}'] = ta.sma(df["Close"], length=period).round(2)
df.tail()

,Date,Open,High,Low,Close,Volume,pct_change,Lag1,Lag2,Lag3,MA_9,MA_21,MA_50,MA_150,MA_200
5418,2026-01-29,335.01,336.00,320.16,321.81,10913390.0,-0.04,333.56,330.12,327.47,329.72,311.83,286.46,266.14,252.09
5419,2026-01-30,325.00,329.50,318.60,323.93,6707829.0,0.01,321.81,333.56,330.12,329.15,313.55,287.83,266.81,252.64
5420,2026-02-02,323.79,330.98,320.26,328.97,5477804.0,0.02,323.93,321.81,333.56,328.61,315.20,289.33,267.48,253.22
5421,2026-02-03,328.97,332.50,327.00,329.43,3603595.0,0.00,328.97,323.93,321.81,328.40,316.82,290.90,268.16,253.80
5422,2026-02-04,329.02,331.00,326.00,327.25,1103130.0,-0.01,329.43,328.97,323.93,327.82,318.43,292.46,268.81,254.37


### 4. Relative Strength Index (RSI)

In [45]:
periods = config.get("features", {}).get("rsi", [14])
for period in periods:
    df[f"RSI_{period}"] = ta.rsi(close=df["Close"], length=period)
df.head()

,Date,Open,High,Low,Close,Volume,pct_change,Lag1,Lag2,Lag3,MA_9,MA_21,MA_50,MA_150,MA_200,RSI_14
0,2004-01-19,52.60,53.40,52.00,52.05,31456000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2004-01-20,52.05,51.85,51.00,51.45,13285500.0,-0.01,52.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
2,2004-01-21,51.45,53.40,51.55,53.10,46880000.0,0.03,51.45,52.05,NaN,NaN,NaN,NaN,NaN,NaN,17.460317
3,2004-01-22,53.10,53.85,53.05,53.10,32411100.0,0.00,53.10,51.45,52.05,NaN,NaN,NaN,NaN,NaN,17.460317
4,2004-01-23,53.10,53.40,52.50,52.65,19969400.0,-0.01,53.10,53.10,51.45,NaN,NaN,NaN,NaN,NaN,16.546506


### 5. Target Variable (next day Close price)

In [46]:
target_col = config.get("model", {}).get("target_column", "Close")
df["Target"] = df[target_col].shift(periods=-1, fill_value=0)
df.head()

,Date,Open,High,Low,Close,Volume,pct_change,Lag1,Lag2,Lag3,MA_9,MA_21,MA_50,MA_150,MA_200,RSI_14,Target
0,2004-01-19,52.60,53.40,52.00,52.05,31456000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.45
1,2004-01-20,52.05,51.85,51.00,51.45,13285500.0,-0.01,52.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,53.10
2,2004-01-21,51.45,53.40,51.55,53.10,46880000.0,0.03,51.45,52.05,NaN,NaN,NaN,NaN,NaN,NaN,17.460317,53.10
3,2004-01-22,53.10,53.85,53.05,53.10,32411100.0,0.00,53.10,51.45,52.05,NaN,NaN,NaN,NaN,NaN,17.460317,52.65
4,2004-01-23,53.10,53.40,52.50,52.65,19969400.0,-0.01,53.10,53.10,51.45,NaN,NaN,NaN,NaN,NaN,16.546506,50.95


### 6. Remove NaN rows

These NaN rows are created during feature engineering

In [47]:
print('Shape After:', df.shape)
df = df.dropna(ignore_index=True)
print('Shape Before:', df.shape)

Shape After: (5423, 17)
Shape Before: (5224, 17)


In [48]:
df.head()

,Date,Open,High,Low,Close,Volume,pct_change,Lag1,Lag2,Lag3,MA_9,MA_21,MA_50,MA_150,MA_200,RSI_14,Target
0,2004-11-03,65.80,66.70,65.65,65.80,26661100.0,0.0,65.80,64.85,65.75,66.14,66.18,64.11,65.49,62.68,52.403501,66.05
1,2004-11-04,65.80,66.45,65.70,66.05,15084300.0,0.0,65.80,65.80,64.85,66.11,66.20,64.11,65.51,62.75,53.778206,66.20
2,2004-11-05,66.05,66.20,65.70,66.20,7516600.0,0.0,66.05,65.80,65.80,66.00,66.24,64.12,65.52,62.83,54.625018,66.25
3,2004-11-08,66.20,66.50,66.00,66.25,9065100.0,0.0,66.20,66.05,65.80,65.97,66.27,64.14,65.54,62.89,54.921482,66.30
4,2004-11-10,66.25,66.50,66.05,66.30,11718300.0,0.0,66.25,66.20,66.05,65.93,66.29,64.16,65.55,62.96,55.236450,66.60


# Train Model

### 1. Prepare X, Y Features

In [49]:
test_size = config.get('model', {}).get('test_size', 0.2)
shuffle = config.get('model', {}).get('shuffle', False)

# features
X = df.drop(columns=["Target", "Date"])

# target
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    shuffle=shuffle
    )
print(X_train.shape)
print(X_test.shape)

(4179, 15)
(1045, 15)


### 2. Model Pipeline

In [50]:
pipe = make_pipeline(
    StandardScaler(),
    LinearRegression()
)
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


### 3. Save Model

In [51]:
# path="../models/linear_regression_pipeline.pkl"
# path = os.path.abspath(path)
# with open(path, "wb") as f:
#     pickle.dump(pipe, f)

### 4. Prediction

In [52]:
def predict(X):
        pred = pipe.predict(X)
        return pred

In [53]:
predict(X_test)

array([ 84.00096529,  81.7035564 ,  82.1832433 , ..., 329.22271854,
       328.83690033, 326.38232336], shape=(1045,))

# Model Evaluation

In [54]:
def calculate_metrics(X, y):
        pred = predict(X)
        mae = mean_absolute_error(y, pred)
        mse = mean_squared_error(y, pred)
        rmse = round(np.sqrt(mse).item(), 4)
        mape = mean_absolute_percentage_error(y, pred)
        r2 = r2_score(y, pred)

        actual_direction = np.sign(np.diff(y))
        predicted_direction = np.sign(np.diff(pred))
        directional_accuracy = round(
            np.mean(actual_direction == predicted_direction).item(), 4)
        metrics = {
            "MAE": [mae],
            "MSE": [mse],
            "RMSE": [rmse],
            "MAPE": [mape],
            "R2 Score": [r2],
            'Directional Accuracy': [directional_accuracy]
        }
        return metrics

In [55]:
def evaluate_train_test(X_train, X_test, y_train, y_test):
        eval_train = pd.DataFrame(calculate_metrics(X=X_train, y=y_train), index=["Train"])
        eval_test = pd.DataFrame(calculate_metrics(X=X_test, y=y_test), index=["Test"])
        concat_evals = pd.concat((eval_train, eval_test), axis='index')
        return concat_evals

In [56]:
evaluate_train_test(X_train, X_test, y_train, y_test)

,MAE,MSE,RMSE,MAPE,R2 Score,Directional Accuracy
Train,1.791287,6.304989,2.5110,1.320476e-02,0.997075,0.5050
Test,2.546804,113.684980,10.6623,1.406598e+15,0.976559,0.5077
